In [1]:
import os
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)

SAMPLE_SUB_PATHS = [
    "/kaggle/input/sample_submission.csv",
    "/kaggle/data/sample_submission.csv",
    "../input/sample_submission.csv",
]

TRAIN_LABELS_PATHS = [
    "/kaggle/input/train_labels.csv",
    "/kaggle/data/train_labels.csv",
    "../input/train_labels.csv",
]

OLD_TRAIN_LABELS_PATHS = [
    "/kaggle/input/old_leaky_data/train_labels_old.csv",
    "/kaggle/data/old_leaky_data/train_labels_old.csv",
    "../input/old_leaky_data/train_labels_old.csv",
]

TRAIN_DIR_CANDIDATES = [
    "/kaggle/input/train",
    "/kaggle/data/train",
    "../input/train",
]

TEST_DIR_CANDIDATES = [
    "/kaggle/input/test",
    "/kaggle/data/test",
    "../input/test",
]

OLD_TRAIN_DIR_CANDIDATES = [
    "/kaggle/input/old_leaky_data/train_old",
    "/kaggle/data/old_leaky_data/train_old",
    "../input/old_leaky_data/train_old",
]


def _read_first_existing_csv(paths):
    for p in paths:
        if os.path.exists(p):
            return pd.read_csv(p)
    raise FileNotFoundError(f"Could not find required CSV in: {paths}")


def _find_first_existing_dir(candidates):
    for d in candidates:
        if os.path.exists(d) and os.path.isdir(d):
            return d
    raise FileNotFoundError(f"Could not find required directory in: {candidates}")


def _read_sample_submission():
    df = _read_first_existing_csv(SAMPLE_SUB_PATHS)
    if not (set(df.columns) >= {"id", "target"}):
        raise ValueError(
            f"sample_submission must contain id,target. Got {df.columns.tolist()}"
        )
    return df[["id", "target"]].copy()


sample_sub = _read_sample_submission()


def align_on_id(df, ref_ids):
    """Align df to ref_ids order by id; requires all ids present."""
    out = df.set_index("id").reindex(ref_ids)
    if out["target"].isna().any():
        missing = int(out["target"].isna().sum())
        raise ValueError(
            f"Alignment produced {missing} missing targets; id sets do not match."
        )
    return out.reset_index()


_HEX_PREFIXES = tuple("0123456789abcdef")


_DIR_FILESET_CACHE = {}


def _get_dir_fileset(d):
    fs = _DIR_FILESET_CACHE.get(d)
    if fs is None:
        try:
            fs = set(os.listdir(d))
        except Exception:
            fs = set()
        _DIR_FILESET_CACHE[d] = fs
    return fs


def _resolve_npy_path(root_dir, id_):
    d0 = os.path.join(root_dir, id_[0])
    fn = f"{id_}.npy"
    if fn in _get_dir_fileset(d0):
        return os.path.join(d0, fn)
    for h in _HEX_PREFIXES:
        d = os.path.join(root_dir, h)
        if fn in _get_dir_fileset(d):
            return os.path.join(d, fn)
    raise FileNotFoundError(f"Could not find .npy for id={id_} under {root_dir}")


def _extract_features_from_path(npy_path):
    """
    Same feature computations (no approximations), kept identical for correctness.
    Input array shape: (6, 273, 256)
    """
    x = np.load(npy_path, mmap_mode="r")  # stored float16, memmap read-only

    a = x[[0, 2, 4]]
    off = x[[1, 3, 5]]

    a32 = a.astype(np.float32, copy=False)
    off32 = off.astype(np.float32, copy=False)

    n = a32.size  # equals off32.size
    inv_n = 1.0 / float(n)

    a_sum = float(a32.sum(dtype=np.float32))
    off_sum = float(off32.sum(dtype=np.float32))
    a_sumsq = float(np.square(a32, dtype=np.float32).sum(dtype=np.float32))
    off_sumsq = float(np.square(off32, dtype=np.float32).sum(dtype=np.float32))

    a_mean = a_sum * inv_n
    off_mean = off_sum * inv_n

    a_var = a_sumsq * inv_n - a_mean * a_mean
    off_var = off_sumsq * inv_n - off_mean * off_mean
    if a_var < 0.0:
        a_var = 0.0
    if off_var < 0.0:
        off_var = 0.0

    a_std = float(np.sqrt(a_var, dtype=np.float32))
    off_std = float(np.sqrt(off_var, dtype=np.float32))

    a_abs = float(np.abs(a32).mean(dtype=np.float32))
    off_abs = float(np.abs(off32).mean(dtype=np.float32))

    a_mean_0 = a32.mean(axis=0, dtype=np.float32)
    off_mean_0 = off32.mean(axis=0, dtype=np.float32)
    diff = a_mean_0 - off_mean_0

    diff_mean = float(diff.mean(dtype=np.float32))
    diff_sumsq = float(np.square(diff, dtype=np.float32).sum(dtype=np.float32))
    diff_n = diff.size
    diff_inv_n = 1.0 / float(diff_n)
    diff_var = diff_sumsq * diff_inv_n - diff_mean * diff_mean
    if diff_var < 0.0:
        diff_var = 0.0
    diff_std = float(np.sqrt(diff_var, dtype=np.float32))
    diff_abs = float(np.abs(diff).mean(dtype=np.float32))

    a_max = float(a32.max())
    off_max = float(off32.max())
    max_gap = a_max - off_max

    eps = 1e-6
    std_ratio = (a_std + eps) / (off_std + eps)
    abs_ratio = (a_abs + eps) / (off_abs + eps)

    a_t = a32.mean(axis=(0, 2), dtype=np.float32)  # (273,)
    off_t = off32.mean(axis=(0, 2), dtype=np.float32)
    dt = a_t - off_t
    dt_mean = float(dt.mean(dtype=np.float32))
    dt_std = float(dt.std(dtype=np.float32))
    dt_abs = float(np.abs(dt).mean(dtype=np.float32))
    dt_max = float(dt.max(initial=-np.inf))

    a_f = a32.mean(axis=(0, 1), dtype=np.float32)  # (256,)
    off_f = off32.mean(axis=(0, 1), dtype=np.float32)
    df = a_f - off_f
    df_mean = float(df.mean(dtype=np.float32))
    df_std = float(df.std(dtype=np.float32))
    df_abs = float(np.abs(df).mean(dtype=np.float32))
    df_max = float(df.max(initial=-np.inf))

    def _quantile_linear_flat(arr, q):
        flat = arr.reshape(-1)
        N = flat.size
        if N == 0:
            return np.nan
        pos = q * (N - 1)
        lo = int(np.floor(pos))
        hi = int(np.ceil(pos))
        if lo == hi:
            kth = np.partition(flat, lo)[lo]
            return float(kth)
        part = np.partition(flat, hi)
        x_lo = float(part[lo])
        x_hi = float(part[hi])
        w = float(pos - lo)
        return x_lo * (1.0 - w) + x_hi * w

    a_p99 = float(_quantile_linear_flat(a32, 0.99))
    off_p99 = float(_quantile_linear_flat(off32, 0.99))
    p99_gap = a_p99 - off_p99

    return np.array(
        [
            a_mean,
            off_mean,
            a_std,
            off_std,
            a_abs,
            off_abs,
            diff_mean,
            diff_std,
            diff_abs,
            a_max,
            off_max,
            max_gap,
            std_ratio,
            abs_ratio,
            dt_mean,
            dt_std,
            dt_abs,
            dt_max,
            df_mean,
            df_std,
            df_abs,
            df_max,
            a_p99,
            off_p99,
            p99_gap,
        ],
        dtype=np.float32,
    )


def _features_cache_path(root_dir, ids, n_features=25):
    root_dir = os.path.abspath(root_dir)
    cache_dir = "/kaggle/working"
    os.makedirs(cache_dir, exist_ok=True)
    ids0 = ids[0] if len(ids) else "none"
    ids1 = ids[-1] if len(ids) else "none"
    h = 2166136261
    for s in (ids0, ids1, str(len(ids))):
        for b in s.encode("utf-8"):
            h ^= b
            h = (h * 16777619) & 0xFFFFFFFF
    key = f"feats_{os.path.basename(root_dir)}_{len(ids)}_{ids0}_{ids1}_{h}_{n_features}.npz"
    return os.path.join(cache_dir, key)


def _build_id_to_path_map(root_dir):
    id_to_path = {}
    for h in _HEX_PREFIXES:
        d = os.path.join(root_dir, h)
        if not (os.path.exists(d) and os.path.isdir(d)):
            continue
        for fn in os.listdir(d):
            if fn.endswith(".npy"):
                id_to_path[fn[:-4]] = os.path.join(d, fn)
    return id_to_path


def _compute_features_parallel(paths, n_features=25):
    from concurrent.futures import ThreadPoolExecutor

    feats = np.empty((len(paths), n_features), dtype=np.float32)

    def _work(i_p):
        i, p = i_p
        return i, _extract_features_from_path(p)

    max_workers = min(32, (os.cpu_count() or 4))
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for i, f in ex.map(_work, enumerate(paths), chunksize=256):
            feats[i] = f
    return feats


def _build_feature_df(root_dir, ids):
    n_features = 25
    cache_path = _features_cache_path(root_dir, ids, n_features=n_features)
    if os.path.exists(cache_path):
        try:
            z = np.load(cache_path)
            feats = z["feats"]
            if feats.shape == (len(ids), n_features):
                cols = [
                    "a_mean",
                    "off_mean",
                    "a_std",
                    "off_std",
                    "a_abs",
                    "off_abs",
                    "diff_mean",
                    "diff_std",
                    "diff_abs",
                    "a_max",
                    "off_max",
                    "max_gap",
                    "std_ratio",
                    "abs_ratio",
                    "dt_mean",
                    "dt_std",
                    "dt_abs",
                    "dt_max",
                    "df_mean",
                    "df_std",
                    "df_abs",
                    "df_max",
                    "a_p99",
                    "off_p99",
                    "p99_gap",
                ]
                return pd.DataFrame(feats, columns=cols)
        except Exception:
            pass  # fall through to recompute

    id_to_path = _build_id_to_path_map(root_dir)
    try:
        paths = [id_to_path[id_] for id_ in ids]
    except KeyError:
        paths = [_resolve_npy_path(root_dir, id_) for id_ in ids]

    feats = _compute_features_parallel(paths, n_features=n_features)

    try:
        np.savez_compressed(cache_path, feats=feats)
    except Exception:
        pass

    cols = [
        "a_mean",
        "off_mean",
        "a_std",
        "off_std",
        "a_abs",
        "off_abs",
        "diff_mean",
        "diff_std",
        "diff_abs",
        "a_max",
        "off_max",
        "max_gap",
        "std_ratio",
        "abs_ratio",
        "dt_mean",
        "dt_std",
        "dt_abs",
        "dt_max",
        "df_mean",
        "df_std",
        "df_abs",
        "df_max",
        "a_p99",
        "off_p99",
        "p99_gap",
    ]
    return pd.DataFrame(feats, columns=cols)




In [2]:
def _local_model_predictions(C=2.0):
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score

    labels_new = _read_first_existing_csv(TRAIN_LABELS_PATHS)
    if not (set(labels_new.columns) >= {"id", "target"}):
        raise ValueError(
            f"train_labels must contain id,target. Got {labels_new.columns.tolist()}"
        )
    labels_new = labels_new[["id", "target"]].copy()
    labels_new["target"] = labels_new["target"].astype(int)

    train_dir_new = _find_first_existing_dir(TRAIN_DIR_CANDIDATES)
    test_dir = _find_first_existing_dir(TEST_DIR_CANDIDATES)

    test_ids = sample_sub["id"].tolist()

    X_all = _build_feature_df(train_dir_new, labels_new["id"].tolist())
    y_all = labels_new["target"].values

    # Change (score-relevant, minimal): use liblinear for this small tabular feature set.
    # This keeps the same LogisticRegression model family and probabilities, but often fits
    # a more useful separator than lbfgs here, improving AUC when the fallback is the only signal.
    base_clf = Pipeline(
        steps=[
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            (
                "lr",
                LogisticRegression(
                    max_iter=600,
                    solver="liblinear",
                    random_state=0,
                    class_weight="balanced",
                    C=float(C),
                ),
            ),
        ]
    )

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    temps = np.array([0.7, 0.85, 1.0, 1.15, 1.3], dtype=np.float64)
    fold_aucs = np.zeros((len(temps),), dtype=np.float64)

    eps = 1e-9
    for tr_idx, va_idx in skf.split(X_all, y_all):
        clf = base_clf
        clf.fit(X_all.iloc[tr_idx], y_all[tr_idx])
        p_va = clf.predict_proba(X_all.iloc[va_idx])[:, 1].astype(np.float64)
        p_va = np.clip(p_va, eps, 1.0 - eps)
        logits_va = np.log(p_va / (1.0 - p_va))

        for i, t in enumerate(temps):
            p_cal = 1.0 / (1.0 + np.exp(-logits_va / t))
            fold_aucs[i] += roc_auc_score(y_all[va_idx], p_cal)

    fold_aucs /= skf.get_n_splits()
    best_t = float(temps[int(np.argmax(fold_aucs))])
    best_auc = float(np.max(fold_aucs))

    base_clf.fit(X_all, y_all)
    X_test = _build_feature_df(test_dir, test_ids)
    proba = base_clf.predict_proba(X_test)[:, 1].astype(np.float64)
    proba = np.clip(proba, eps, 1.0 - eps)
    test_logits = np.log(proba / (1.0 - proba))
    proba_cal = 1.0 / (1.0 + np.exp(-test_logits / best_t))

    out = pd.DataFrame({"id": test_ids, "target": proba_cal})
    out["target"] = out["target"].clip(0.0, 1.0)
    return out, {"C": float(C), "best_t": best_t, "oof_auc": best_auc}


# Change (score-relevant, minimal): pick the best C by CV AUC rather than a fixed default.
# This keeps the same model class and training approach (5-fold CV + refit), but avoids
# ending up with an effectively constant ~0.5 predictor that yields ~0.5 AUC.
def _local_model_predictions_autoC():
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline

    labels_new = _read_first_existing_csv(TRAIN_LABELS_PATHS)[["id", "target"]].copy()
    labels_new["target"] = labels_new["target"].astype(int)

    train_dir_new = _find_first_existing_dir(TRAIN_DIR_CANDIDATES)
    X_all = _build_feature_df(train_dir_new, labels_new["id"].tolist())
    y_all = labels_new["target"].values

    Cs = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

    best_C = Cs[0]
    best_auc = -1.0

    for C in Cs:
        clf = Pipeline(
            steps=[
                ("scaler", StandardScaler(with_mean=True, with_std=True)),
                (
                    "lr",
                    LogisticRegression(
                        max_iter=600,
                        solver="liblinear",
                        random_state=0,
                        class_weight="balanced",
                        C=float(C),
                    ),
                ),
            ]
        )

        oof = np.zeros_like(y_all, dtype=np.float64)
        for tr_idx, va_idx in skf.split(X_all, y_all):
            clf.fit(X_all.iloc[tr_idx], y_all[tr_idx])
            oof[va_idx] = clf.predict_proba(X_all.iloc[va_idx])[:, 1].astype(np.float64)

        auc = float(roc_auc_score(y_all, oof))
        if auc > best_auc:
            best_auc = auc
            best_C = C

    pred, meta = _local_model_predictions(C=best_C)
    meta = dict(meta)
    meta["autoC_oof_auc"] = best_auc
    return pred, meta


_LOCAL_FALLBACK_DF = None
_LOCAL_FALLBACK_META = None


def read_submission_or_fallback(path):
    """
    Try reading a submission CSV from `path`. If not found, use local-model fallback.
    Always returns columns: id,target with float target aligned to sample_submission ids.
    """
    global _LOCAL_FALLBACK_DF, _LOCAL_FALLBACK_META
    ref_ids = sample_sub["id"].tolist()

    if path is not None and os.path.exists(path):
        df = pd.read_csv(path)
        if not (set(df.columns) >= {"id", "target"}):
            raise ValueError(
                f"Submission at {path} must contain columns id,target. Got {df.columns.tolist()}"
            )
        df = df[["id", "target"]].copy()
        df["target"] = df["target"].astype(float)
        return align_on_id(df, ref_ids)

    if _LOCAL_FALLBACK_DF is None:
        pred, meta = _local_model_predictions_autoC()
        _LOCAL_FALLBACK_DF = align_on_id(pred, ref_ids)
        _LOCAL_FALLBACK_META = meta
    return _LOCAL_FALLBACK_DF.copy()


def submission_exists(path):
    """Used to adapt weights: True if a referenced upstream submission file exists on disk."""
    return path is not None and os.path.exists(path)




In [3]:
P1 = "../input/rerun-seti-e-t-volo-d1-baseline-inference/submission.csv"
P2 = "../input/seti-bl-spatial-info-tf-tpu/submission.csv"
P3 = "../input/seti-bl-tf-starter-tpu/submission.csv"
P4 = "../input/seti-learned-image-resizing/submission.csv"
P5 = "../input/lb-0-980-efficientnet-b0-more-epoch/submission.csv"
P6 = "../input/inference-5x-ensemble-vanilla-resnet34d-seti/submission.csv"

data1 = read_submission_or_fallback(P1)
data2 = read_submission_or_fallback(P2)
data3 = read_submission_or_fallback(P3)
data4 = read_submission_or_fallback(P4)
data5 = read_submission_or_fallback(P5)
data6 = read_submission_or_fallback(P6)



In [4]:
ref_ids = sample_sub["id"].tolist()



In [5]:
base_weights = {
    "data5": 0.795,
    "data4": 0.1025,
    "data6": 0.1025,
    "data1": 0.0,
    "data2": 0.0,
    "data3": 0.0,
}

available_disk = {
    "data1": submission_exists(P1),
    "data2": submission_exists(P2),
    "data3": submission_exists(P3),
    "data4": submission_exists(P4),
    "data5": submission_exists(P5),
    "data6": submission_exists(P6),
}

if not any(available_disk.values()):
    fb_specs = {
        "data4": 0.5,
        "data5": 2.0,
        "data6": 8.0,
    }
    metas = {}
    preds = {}
    for k, C in fb_specs.items():
        pred, meta = _local_model_predictions(C=C)
        preds[k] = align_on_id(pred, ref_ids)
        metas[k] = meta

    data4, data5, data6 = preds["data4"], preds["data5"], preds["data6"]

    ranked = sorted(
        [(metas[k]["oof_auc"], k) for k in ["data4", "data5", "data6"]], reverse=True
    )
    best_k = ranked[0][1]
    for k in ["data4", "data5", "data6"]:
        base_weights[k] = 0.15
    base_weights[best_k] = 0.70
    base_weights["data1"] = 0.0
    base_weights["data2"] = 0.0
    base_weights["data3"] = 0.0

if available_disk["data2"] or available_disk["data3"] or available_disk["data1"]:
    extra_total = 0.05
    extras = []
    if available_disk["data1"]:
        extras.append("data1")
    if available_disk["data2"]:
        extras.append("data2")
    if available_disk["data3"]:
        extras.append("data3")

    per_extra = extra_total / len(extras)
    base_weights["data5"] = max(0.0, base_weights["data5"] - extra_total)
    for k in extras:
        base_weights[k] = per_extra

main_keys = ["data5", "data4", "data6"]
missing_weight = 0.0
for k in ["data1", "data2", "data3"]:
    if not available_disk[k]:
        missing_weight += float(base_weights.get(k, 0.0))
        base_weights[k] = 0.0

if missing_weight > 0:
    add = missing_weight / len(main_keys)
    for k in main_keys:
        base_weights[k] += add

dfs = {
    "data1": data1,
    "data2": data2,
    "data3": data3,
    "data4": data4,
    "data5": data5,
    "data6": data6,
}

group_rep = {}
rep_for_key = {}
for k, df in dfs.items():
    arr = np.asarray(df["target"].values, dtype=np.float64)
    sig = (
        arr.shape[0],
        float(arr.mean()),
        float(arr.std()),
        float(np.round(arr[:64], 12).sum()),
    )
    if sig in group_rep:
        rep_for_key[k] = group_rep[sig]
    else:
        group_rep[sig] = k
        rep_for_key[k] = k

agg_w = {}
for k, w in base_weights.items():
    rep = rep_for_key.get(k, k)
    agg_w[rep] = agg_w.get(rep, 0.0) + float(w)

w_sum = float(sum(agg_w.values()))
weights = {k: (v / w_sum if w_sum > 0 else 0.0) for k, v in agg_w.items()}

blend = pd.DataFrame(
    {"id": ref_ids, "target": np.zeros(len(ref_ids), dtype=np.float64)}
)
for k, w in weights.items():
    blend["target"] += w * dfs[k]["target"].values.astype(np.float64)

blend["target"] = blend["target"].clip(0.0, 1.0)



In [6]:
submission = blend[["id", "target"]].copy()
submission.to_csv("submission.csv", index=False)

print("Wrote submission.csv with shape:", submission.shape)
print("Blend weights used (after dedup):", weights)
print(submission.head())

Wrote submission.csv with shape: (6000, 2)
Blend weights used (after dedup): {'data5': 0.15, 'data4': 0.7, 'data6': 0.15, 'data1': 0.0}
                id    target
0  0cee567456cd304  0.487327
1  5451b45281c65a7  0.488036
2  f8cc6cea820282d  0.498241
3  25e21ba81a64742  0.500219
4  aafa910406b1db2  0.524727
